# Lab · Đạo hàm tự động (autograd) · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho phần B của [`lesson.md`](../lesson.md) (Bài 2). Mỗi bài có đề, 3 tầng gợi ý (mở dần), ô code để bạn viết và ô kiểm tra tự động.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.
Mẹo: mỗi bài hãy tính đạo hàm **bằng tay trước**, rồi dùng PyTorch kiểm tra.

Môi trường Python do **uv** quản lý. Trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi mở lab bằng `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. Đạo hàm đầu tiên: y = x²

Tạo `x` là scalar `2.0` **có theo dõi gradient**, tính `y = x²`, rồi gọi `backward()`.
Lưu `x.grad` vào `grad1`. Tính bằng tay trước: dy/dx tại x = 2 là bao nhiêu?

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
grad1 = x.grad
print(grad1)

In [ ]:
assert grad1.item() == 4.0, "dy/dx = 2x = 4 tại x = 2"
print("✅ Bài 1 đúng")

## Bài 2. Hàm phức tạp hơn: y = x² + 2x + 1

Cùng `x = 2.0`. Tính `y = x² + 2x + 1`, `backward()`, lưu `grad2 = x.grad`.
Tính bằng tay: dy/dx = 2x + 2 nên bằng bao nhiêu tại x = 2?
Sau đó lưu `gf = type(y.grad_fn).__name__` để xem nút cuối của đồ thị là phép toán gì.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()
grad2 = x.grad
gf = type(y.grad_fn).__name__
print(grad2, gf)

In [ ]:
assert grad2.item() == 6.0
assert "Add" in gf, "nút cuối của y là một phép cộng"
print("✅ Bài 2 đúng")

## Bài 3. Tự làm: y = 2x³ + x tại x = 1

Tính bằng tay đạo hàm của `y = 2x³ + x` tại `x = 1`, rồi dùng autograd kiểm tra. Lưu vào `grad3`.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = 2 * x ** 3 + x
y.backward()
grad3 = x.grad
print(grad3)

In [ ]:
assert grad3.item() == 7.0
print("✅ Bài 3 đúng")

## Bài 4. Đạo hàm riêng

- (a) `f(u, v) = u²v + v³` tại `u = 3`, `v = 1`. Một lần `backward()`, lấy `du = u.grad`, `dv = v.grad`.
  Tính bằng tay: ∂f/∂u = 2uv, ∂f/∂v = u² + 3v².
- (b) Với hàm trong (a), vì sao chỉ cần **một** lần `backward()` mà có cả hai đạo hàm?

In [ ]:
u = torch.tensor(3.0, requires_grad=True)
v = torch.tensor(1.0, requires_grad=True)
f = u ** 2 * v + v ** 3
f.backward()
du = u.grad
dv = v.grad
print(du, dv)

In [ ]:
assert du.item() == 6.0, "∂f/∂u = 2uv = 6"
assert dv.item() == 12.0, "∂f/∂v = u² + 3v² = 9 + 3 = 12"
print("✅ Bài 4 đúng")

## Bài 5. Gradient cộng dồn

`x = 2.0` có theo dõi gradient. Lặp **3 lần**: mỗi lần tính lại `y = x²`, `backward()`, rồi thêm `x.grad.item()` vào danh sách `acc`.
Đoán trước `acc` sẽ là gì.
Sau vòng lặp, đặt gradient về 0 bằng `x.grad.zero_()`, rồi làm thêm **một** lượt (tính y, backward) và lưu `x.grad.item()` vào `after_reset`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
acc = []
for _ in range(3):
    y = x ** 2
    y.backward()
    acc.append(x.grad.item())
x.grad.zero_()
y = x ** 2
y.backward()
after_reset = x.grad.item()
print(acc, after_reset)

In [ ]:
assert acc == [4.0, 8.0, 12.0], "gradient cộng dồn: 4, 4+4, 4+4+4"
assert after_reset == 4.0
print("✅ Bài 5 đúng: trong huấn luyện, nhớ đặt gradient về 0 mỗi lượt")

## Bài 6. Đầu vào là vector: gradient của `sum`

`x = torch.linspace(-3, 3, 7)` có theo dõi gradient, `y = x²` (vector 7 phần tử).
- Thử `y.backward()` trong `try/except RuntimeError`; lưu thông báo vào `err_msg`.
- Gộp `y` thành một số bằng `sum`, `backward()`, lưu `x.grad` vào `gx`.
- Vẽ `y` theo `x` và `gx` theo `x` trên cùng một đồ thị. Nhớ `detach().numpy()`.

In [ ]:
x = torch.linspace(-3, 3, 7, requires_grad=True)
y = x ** 2

err_msg = None
try:
    y.backward()
except RuntimeError as e:
    err_msg = str(e)

y.sum().backward()
gx = x.grad

plt.plot(x.detach().numpy(), y.detach().numpy(), label="y = x²")
plt.plot(x.detach().numpy(), gx.detach().numpy(), label="dy/dx")
plt.legend()
plt.show()

In [ ]:
assert err_msg is not None and "scalar" in err_msg
assert torch.allclose(gx, 2 * x.detach()), "dy/dx = 2x"
print("✅ Bài 6 đúng")

## Bài 7. Quy tắc chuỗi

`z = (3x + 1)²` tại `x = 1`. Đây là hàm lồng hàm: hàm ngoài `g(u) = u²`, hàm trong `u = 3x + 1`.
- `manual`: tính đạo hàm **bằng công thức chain rule** `g'(u) · u'(x)` ở x = 1 (dưới dạng số Python).
- `auto`: đạo hàm do autograd tính.
Hai giá trị phải bằng nhau.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
z = (3 * x + 1) ** 2
z.backward()
auto = x.grad.item()
manual = 2 * (3 * 1 + 1) * 3
print(auto, manual)

In [ ]:
assert auto == manual == 24.0
print("✅ Bài 7 đúng")

## Bài 8. Đạo hàm của ReLU

`r = torch.tensor([-2., -1., 0., 1., 2.])` có theo dõi gradient. ReLU là `torch.relu(r)` (= max(0, r)).
Gộp bằng `sum`, `backward()`, lưu `r.grad` vào `grelu`. Đoán trước mẫu hình của `grelu`, đặc biệt tại r = 0.

In [ ]:
r = torch.tensor([-2., -1., 0., 1., 2.], requires_grad=True)
torch.relu(r).sum().backward()
grelu = r.grad
print(grelu)

In [ ]:
assert grelu.tolist() == [0., 0., 0., 1., 1.]
print("✅ Bài 8 đúng: PyTorch quy ước đạo hàm tại 0 là 0")

## Bài 9. Thử thách: tự cập nhật tham số bằng gradient

Tìm `w` để `loss = (w - 3)²` nhỏ nhất, bằng cách đi ngược hướng gradient:
1. `w = 0.0` có theo dõi gradient.
2. Lặp 25 lần: tính `loss`, `loss.backward()`, cập nhật `w ← w − 0.1 · w.grad`, rồi đặt `w.grad` về 0.
   Việc cập nhật `w` **không** được ghi vào đồ thị tính toán (bọc trong `with torch.no_grad():`).
3. Lưu giá trị cuối vào `w_final` (số Python).

Đây là gradient descent, nền tảng của module 3.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1
for step in range(25):
    loss = (w - 3) ** 2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    w.grad.zero_()
w_final = w.item()
print(w_final)

In [ ]:
assert abs(w_final - 3.0) < 0.05, f"w nên gần 3, đang là {w_final:.3f}"
print("✅ Bài 9 đúng: bạn vừa viết xong vòng lặp huấn luyện đầu tiên")

## Hết